In [1]:
!pip install -q google-genai pydantic

In [ ]:
# Task 4.1 - one validated call to Gemini
import json
import time
import pandas as pd
from google import genai
from google.genai import types

MODEL_NAME = "gemini-3.5-flash"   # use the Flash model name from your Session 22.1 notebook

from getpass import getpass

API_KEY = getpass("Paste your key: ")   # hidden while typing, so it never shows in the output
if API_KEY.strip() == "":
    raise ValueError("No API key entered")
client = genai.Client(api_key=API_KEY)

ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
           "illness_or_emergency", "work_or_schedule", "other"]

SCHEMA = {
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}

SYSTEM_TEXT = ("You help a chain of outpatient clinics understand why patients miss appointments. "
               "You read short call-centre notes and reply only with JSON.")

api_calls = 0      # counting every call we make (free tier is small)
last_raw = ""      # the exact text of the latest reply


def craft_prompt(note_text):
    # prompt built with the CRAFT parts
    return f"""Context: CareLoop Clinics runs eight outpatient clinics in India. After a patient misses an
appointment, the call centre phones them and writes a short note. The note may be in English or Hinglish.

Role: You are a patient-experience analyst at the clinic.

Action: Read the note and find the main reason the patient missed the visit. Pick root_cause from
exactly one of these values: {", ".join(ALLOWED)}.
If the note does not clearly give a reason (for example the patient could not be reached), answer
"other" instead of guessing. Give a confidence between 0 and 1, a summary of at most 25 words,
and one practical recommended_action for the clinic.

Tone: Neutral and factual.

Format: JSON with the fields root_cause, confidence, summary, recommended_action.

Note: {note_text}"""


def ask_gemini(prompt):
    # sends a prompt and returns the raw text of the reply
    global api_calls, last_raw
    api_calls += 1
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")))  # delete this line if your model rejects it
    last_raw = response.text
    return response.text


def analyse_note(note_text):
    raw = ask_gemini(craft_prompt(note_text))
    return json.loads(raw)


notes = pd.read_csv("followup_notes.csv")
n001 = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]

result = analyse_note(n001)
print(result)
print("API calls so far:", api_calls)

Paste your key: ··········
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.95, 'summary': 'Patient forgot the appointment because she did not receive a reminder message and requested to reschedule.', 'recommended_action': 'Verify patient contact details and check automated appointment reminder delivery logs.'}
API calls so far: 1


In [3]:
# Task 4.2 - validate and process the first 10 notes
from pydantic import BaseModel, Field, ValidationError


class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)


def root_cause_allowed(insight):
    return insight.root_cause in ALLOWED


fields = ["root_cause", "confidence", "summary", "recommended_action"]
rows = []

for i, note in notes.head(10).iterrows():
    note_text = note["note_text"]
    row = {"note_id": note["note_id"], "appointment_id": note["appointment_id"]}

    # blank note -> don't call the API
    if pd.isna(note_text) or str(note_text).strip() == "":
        row.update({"root_cause": "other", "confidence": 0.0, "summary": "No note recorded",
                    "recommended_action": "Re-call patient", "validation_status": "skipped_blank",
                    "raw_response": ""})
        rows.append(row)
        print(note["note_id"], "-> skipped (blank note)")
        continue

    last_raw = ""
    try:
        data = analyse_note(note_text)
        # copy whatever fields came back, exactly as they are
        for f in fields:
            row[f] = data.get(f)
        insight = FollowupInsight(**data)
        if root_cause_allowed(insight):
            row["validation_status"] = "valid"
        else:
            row["validation_status"] = "invalid"
    except Exception as e:
        # bad JSON, failed validation or an API error - keep going
        row["validation_status"] = "invalid"
        print("  problem with", note["note_id"], "->", e)

    row["raw_response"] = last_raw
    rows.append(row)
    print(note["note_id"], "->", row.get("root_cause"), "|", row["validation_status"])

    time.sleep(13)   # stay under the free-tier rate limit

results = pd.DataFrame(rows, columns=["note_id", "appointment_id"] + fields + ["validation_status", "raw_response"])
results.to_csv("notes_analysed.csv", index=False)

print("\nnotes_analysed.csv saved:", results.shape)
print("API calls so far:", api_calls)
results[["note_id", "root_cause", "confidence", "validation_status"]]

N001 -> forgot_or_no_reminder | valid
N002 -> distance_or_transport | valid
N003 -> cost_or_fee | valid
N004 -> skipped (blank note)
N005 -> illness_or_emergency | valid
N006 -> work_or_schedule | valid
N007 -> forgot_or_no_reminder | valid
N008 -> distance_or_transport | valid
N009 -> other | valid
N010 -> work_or_schedule | valid

notes_analysed.csv saved: (10, 8)
API calls so far: 10


,note_id,root_cause,confidence,validation_status
0,N001,forgot_or_no_reminder,0.95,valid
1,N002,distance_or_transport,0.95,valid
2,N003,cost_or_fee,0.95,valid
3,N004,other,0.00,skipped_blank
4,N005,illness_or_emergency,0.95,valid
5,N006,work_or_schedule,0.95,valid
6,N007,forgot_or_no_reminder,0.95,valid
7,N008,distance_or_transport,0.95,valid
8,N009,other,1.00,valid
9,N010,work_or_schedule,0.95,valid


In [4]:
# Task 4.3 - breaking it on purpose

# (a) two bad replies that should fail validation
bad_replies = [
    {"root_cause": "cost_or_fee", "confidence": 1.7,
     "summary": "Patient found the fee too high.", "recommended_action": "Offer a discount"},
    {"root_cause": "cost_or_fee", "confidence": 0.8,
     "summary": "Patient found the fee too high."},   # no recommended_action
]

for bad in bad_replies:
    try:
        FollowupInsight(**bad)
        print("Passed (should not happen)")
    except ValidationError as e:
        print("Caught a validation error:")
        print(e)
        print()

# (b) V1 vs V2 prompt on the two tricky notes
def v1_prompt(note_text):
    return f"Classify this note: {note_text}"

tests = notes[notes["note_id"].isin(["N007", "N009"])]
comparison = []

for _, note in tests.iterrows():
    for version, make_prompt in [("V1", v1_prompt), ("V2", craft_prompt)]:
        try:
            reply = json.loads(ask_gemini(make_prompt(note["note_text"])))
            cause = reply.get("root_cause")
        except Exception as e:
            cause = f"error: {e}"
        comparison.append({"note_id": note["note_id"], "prompt": version,
                           "root_cause": cause, "allowed": cause in ALLOWED})
        time.sleep(13)

comparison = pd.DataFrame(comparison)
print(comparison.to_string(index=False))

# I would deploy V2: it tells the model the exact allowed values and to answer
# "other" instead of guessing, so its answers can be checked and counted.
# V1 has no list, so the model makes up its own labels.

# (c) total API calls
print("\nTotal API calls in Tasks 4.1-4.3:", api_calls)

Caught a validation error:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Caught a validation error:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'cost_or_f...ound the fee too high.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

note_id prompt                                   root_cause  allowed
   N007     V1 Forgotten appointment / Missing SMS reminder    False
   N007     V2                        forgot_or_no_reminder     True
   N009     V1                          Unreachable patient    False
   N009     V2                                        other     True

Total API calls in Tasks 4.1-4.3: 14


In [5]:
# Task 4.4 - join the reasons with the appointment data
analysed = pd.read_csv("notes_analysed.csv")
appts = pd.read_csv("appointments_clean.csv")

merged = analysed.merge(appts, on="appointment_id")
print("Rows after merge:", len(merged))

print(pd.crosstab(merged["root_cause"], merged["reminder_sent"], margins=True))

Rows after merge: 10
reminder_sent          No  Yes  All
root_cause                         
cost_or_fee             0    1    1
distance_or_transport   1    1    2
forgot_or_no_reminder   2    0    2
illness_or_emergency    0    1    1
other                   1    1    2
work_or_schedule        1    1    2
All                     5    5   10
